# Note detection: expanded 449-track evaluation

Compare Attune's current pYIN+ → linear KernelCPD → score alignment and repeat-recovery pipeline with **Basic Pitch**, **CREPE Notes**, **Tony/pYIN Vamp**, **official MT3**, and **YourMT3+ (YPTF.MoE+Multi, noPS)**. External methods keep their native complete transcription pipelines. This is an exploratory, score-informed application comparison, not an audio-only state-of-the-art claim.

Attune receives the supplied MIDI score part (shortest-note prior) and the same reference-F0-derived pitch range as `pitch.ipynb`; the other methods receive audio only. URMP's performed-note `Notes` annotations are used **only by the evaluator** (the separate frame-F0 annotations set the benchmark pitch range); Attune gets the separate `Sco_*.mid` part. Bach10 likewise uses original isolated recordings, supplied score MIDI parts, and separate `GTNotes.mat` performed-note references available only to the evaluator. The optional `attune-audio-only` ablation removes score access and uses fixed range/minimum duration. Attune includes production score alignment, robust timing refitting, and repeat recovery. No reference-based latency adjustment or boundary trimming is performed.

The existing notebooks are `../../notebooks/archive/benchmark_notes.ipynb` (historical PELT), `benchmarks.ipynb` (old combined entry point), and `sweeps/note.ipynb` (parameter selection). See [audit and competitor research](../../docs/archive/note-benchmark-audit-2026-09-23.md) before citing results.

**Run All:** evaluate 449 tracks using the same seed-0 selection as the expanded pitch evaluation: 260 CocoChorales test stems (20 per instrument), all 149 URMP stems, and all 40 original Bach10 stems. All six methods remain selected. Compatible successful note checkpoints are reused automatically; missing or incompatible note results are computed and checkpointed for resumption.

Attune uses the already computed shared smoothed pitch data. Setup checks all 449 pitch caches against the current cache version, smoother version, and process-CPU timing contract before any note jobs launch. Missing/incompatible pitch data stops setup with the affected paths; refresh those tracks in the pitch evaluation first. Keep the pitch caches unchanged while this notebook runs. No pitch inference is needed with compatible caches. Restart the kernel and choose **Run All**. Old outputs have been cleared to avoid presenting the 60-track results as expanded results.


YourMT3+ uses **two persistent model instances, two concurrent recordings per instance** (four recordings total). Completed checkpoints from interrupted runs are reused automatically. New CPU timings use per-request thread CPU plus amortized model startup; cached serial timings retain their original clock labels.


In [1]:
from dataclasses import replace
from pathlib import Path
import sys
REPO_ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / 'app.py').is_file()))
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from IPython.display import display
import pandas as pd
import importlib
for name in ("NoteDetectorBase", "NoteCache", "competitors.Ruptures", "competitors.SlopeWindow",
             "competitors.Attune", "competitors.BasicPitch", "competitors.CrepeNotes", "competitors.Tony",
             "competitors.MT3", "competitors.YourMT3",
             "NoteBenchmarker", "NoteNotebook"):
    importlib.reload(importlib.import_module(f"benchmarks.modules.note.{name}"))
from benchmarks.modules.note.NoteNotebook import NotebookConfig
from benchmarks.modules.note.NoteNotebook import NoteNotebook
from benchmarks.modules.pitch.PitchNotebook import PitchNotebook
PITCH_CONFIG = PitchNotebook.NotebookConfig()
# Production Attune plus the selected external competitors; no ruptures sweep.
CONFIG = NotebookConfig(methods=("attune", "basic-pitch", "crepe-notes", "tony",
                                 "mt3", "yourmt3"),
                        seed=0, workers=PITCH_CONFIG.workers, neural_workers=6,
                        force=False, force_methods=(), use_pitch_cache=True,
                        cache_only_methods=(), max_tracks=None)
# None means all available stems; Coco uses the pitch paper's seeded sample.
SELECTION = {"coco": (20, 260), "urmp": (None, 149), "bach10-original": (None, 40)}
BENCHES = {dataset: NoteNotebook(replace(CONFIG, preliminary_tracks_per_instrument=count))
           for dataset, (count, expected) in SELECTION.items()}
BENCH = BENCHES["coco"]  # Common metric/reporting helpers.
BENCH.show_configuration()

from algorithms.Config import Config as PitchConfig
from algorithms.PitchSmoother import PitchSmoother
from benchmarks.modules.pitch.PitchCache import PitchCache
from benchmarks.modules.pitch.competitors.Attune import Attune as PitchAttune

selected_tracks = {}
pitch_cache_problems = []
for dataset, bench in BENCHES.items():
    tracks = bench.select_tracks(dataset)
    expected = SELECTION[dataset][1]
    if len(tracks) != expected or len({t["track_id"] for t in tracks}) != expected:
        raise RuntimeError(f"{dataset}: expected {expected} unique tracks, got {len(tracks)}")
    selected_tracks[dataset] = tracks
    for track in tracks:
        cache = PitchCache(track["pitch_cache"])
        try:
            stages, metadata = cache._payload()
            smooth = metadata.get(PitchCache.SMOOTHED, {})
            compatible = (
                PitchCache.SMOOTHED in stages
                and smooth.get("pitch_cache_version") == PitchCache.VERSION
                and smooth.get("pitch_smoother_method") == PitchSmoother.METHOD_VERSION
                and cache.has_current_timing(PitchCache.SMOOTHED, PitchAttune.COMPUTE_CLOCK)
                and cache.read(PitchCache.SMOOTHED, PitchConfig()) is not None
            )
            if not compatible:
                raise ValueError("incompatible smoothed pitch data or timing")
        except Exception as exc:
            pitch_cache_problems.append({"dataset": dataset, "track_id": track["track_id"],
                                         "pitch_cache": str(cache.path), "error": str(exc)})
if pitch_cache_problems:
    display(pd.DataFrame(pitch_cache_problems))
    raise RuntimeError("Refresh the listed shared pitch caches in the pitch evaluation, then Run All again.")
display(pd.DataFrame([{"dataset": dataset, "selected_tracks": len(tracks)}
                      for dataset, tracks in selected_tracks.items()]))
print("All 449 shared Attune pitch caches passed compatibility checks.")

# Preview compatible note checkpoints and pending note/transcription jobs.
for dataset, bench in BENCHES.items():
    display(bench.plan_preliminary(dataset).groupby(["dataset", "method", "action"])
            .size().rename("recordings").to_frame())

# Presentation labels only; retain stable metric keys in cached results.
METRIC_LABELS = {"f1": "Onset F1", "f1_offset": "Transcription F1"}
def display_metrics(frame):
    shown = frame.rename(columns=METRIC_LABELS).copy()
    if "metric" in shown.columns:
        shown["metric"] = shown["metric"].replace(METRIC_LABELS)
    for column in ("method", "competitor"):
        if column in shown.columns:
            shown[column] = shown[column].replace({"attune": "Attune"})
    display(shown)


,setting,value
0,workers,8
1,neural_workers,6
2,preliminary_tracks_per_instrument,20
3,seed,0
4,max_tracks,None
5,methods,"(attune, basic-pitch, crepe-notes, tony, mt3, ..."
6,onset_tolerance,0.05
7,pitch_tolerance,50.0
8,offset_ratio,0.2
9,offset_min_tolerance,0.05


,dataset,selected_tracks
0,coco,260
1,urmp,149
2,bach10-original,40


All 449 shared Attune pitch caches passed compatibility checks.


recordings
dataset method      action             
coco    attune      cached          260
        basic-pitch cached          260
        crepe-notes cached          260
        mt3         pending         260
        tony        cached          260
        yourmt3     pending         260

recordings
dataset method      action             
urmp    attune      pending         149
        basic-pitch cached           26
                    pending         123
        crepe-notes cached           26
                    pending         123
        mt3         pending         149
        tony        cached           26
                    pending         123
        yourmt3     pending         149

recordings
dataset         method      action             
bach10-original attune      pending          40
                basic-pitch cached            8
                            pending          32
                crepe-notes cached            8
                            pending          32
                mt3         pending          40
                tony        cached            8
                            pending          32
                yourmt3     pending          40

## Protocol and CPU timing

Seed 0 selects 20 stems per available Coco instrument (260 total), all 149 URMP stems, and all 40 original Bach10 stems using the existing pitch dataset samplers. Setup asserts these counts and unique track identities. Coco uses the test split, original isolated stem audio and stem MIDI. URMP uses isolated `AuSep` recordings and official `Notes` files: onset seconds, frequency Hz, duration seconds. Bach10 uses all 40 original recordings (ten each of violin, clarinet, saxophone and bassoon). Its `GTNotes.mat` references supply performed boundaries on the 10 ms grid: MATLAB frame 1 is centered at 23 ms, and first/last frame centers define onset/offset. Each note is assigned the median fractional MIDI pitch of its annotated contour, converted to Hz without rounding to a score pitch. Supplied score MIDI parts condition Attune separately; interpolated `.txt` score-alignment times are not used as note ground truth. Full recordings are scored. Selection manifests are saved before execution; missing selected inputs fail loudly.

Primary quality: **Transcription F1**, pooled-note (micro) F1 requiring **50 ms onset and 50 cents pitch** tolerance plus offsets within **max(50 ms, 20% of reference duration)**. **Onset F1** uses the same pitch/onset requirements with offsets ignored. Both metrics also report precision and recall. Empty predictions score zero; dependency/runtime failures remain errors, never zeros. Full polyphonic Basic Pitch predictions are retained, including simultaneous onsets.

Jobs run in fresh spawned processes, normally CPU count minus one workers. Heavy neural methods use up to six workers to bound memory. Every numerical runtime gets one thread; GPU execution is disabled. Results include:

| Column | Measured scope |
|---|---|
| `cpu_seconds` | Full algorithm CPU: frontend + note stage; a reused frontend retains its original measured CPU cost. Tony reports its complete call because stages are unavailable |
| `execution_cpu_seconds` | CPU actually spent in this invocation, including imports/setup/cache loading; a cache hit makes this smaller without making the algorithm appear free |
| `frontend_cpu_seconds` | Attune pitch detection+smoothing; Basic Pitch neural inference; CREPE tracking |
| `segmentation_cpu_seconds` | Attune score-span/minimum-length preparation + production KernelCPD; Basic Pitch native note creation; CREPE Notes native post-processing including amplitude analysis and MIDI writing |
| `wall_seconds` | Observed invocation elapsed time; cache- and contention-dependent diagnostic |
| `audio_per_cpu` | Total audio duration divided by total full-call CPU seconds |

CPU time is `process_time` plus reaped child user/system CPU, so Tony's Sonic Annotator work is counted. Tony exposes no separate segmentation timer: stage columns are missing, not zero. Stage boundaries are not identical algorithms: Basic Pitch has already inferred onset/note activations before note creation. Compare stage times with that qualification. CPU timing avoids charging scheduling waits, though contention can still affect CPU efficiency. Stage timings include different native loading/model-initialization costs: Attune follows the pitch notebook’s detector+smoother timing, while the external APIs load audio/models within their frontend. Neither aggregate CPU nor invocation time is steady-state interactive latency.

Attune reuses `pitch.ipynb`’s full smoothed `PitchData` cache, at the same sample rate, range, seed and selected stem IDs. Reuse requires the current cache version, production smoother version and process-CPU timing; the setup preflight stops on misses before launching evaluation. The underlying adapter can regenerate missing pitch data outside this notebook, so do not modify caches during a run. Old wall-clock or incompatible caches are not accepted. These legacy cache files do not contain full source/config fingerprints, so the compatibility guarantee is their version contract, not proof of arbitrary older settings; set `use_pitch_cache=False` to force a fresh frontend. Successful note checkpoints are fingerprinted and resumable. This notebook sets `cache_only_methods=()` so missing or incompatible note checkpoints are computed. The separate setup preflight requires compatible shared Attune pitch caches for all selected tracks. `force_methods=()` (the notebook default) resumes compatible jobs for every method. `force=True` deliberately bypasses reuse, and `force_methods` can explicitly rerun selected methods. Otherwise inputs, inference code, runtime dependencies, scoring settings, and saved success determine the schedule. Each run saves input/source hashes, versions, config, selection, predictions, per-job logs, raw rows, summaries and paired comparisons under `results/note/notebook_runs/`. CREPE Notes reuses the shared CREPE raw frequency/confidence cache (`crepe__<track>.raw.npz`) produced by the pitch evaluation when audio, model, preprocessing, dependency versions, and timing are compatible. Legacy thresholded CREPE estimates alone cannot supply the required confidence; missing or incompatible raw caches trigger CREPE inference and are saved for reuse. Basic Pitch’s pitch cache lacks note/onset activations, so it reuses complete predictions through compatible note-run checkpoints instead.

Live progress uses the exact `PitchBenchmarker.Progress` compact renderer from `pitch.ipynb`: one updating `[   n/total] method: track_id` line across all selected method/track pairs, with the same track IDs. Cached and newly computed tracks each advance this counter once; fully cached runs use the same display and reach the same total. Errors are listed after the live line finishes; full Python/native runtime output stays in each job’s log.


CREPE Notes retains its native CNN onset splitting. Its adapter resolves installed FFmpeg/ffprobe tools even when the notebook PATH omits Homebrew; this fixes the earlier decoder failures on CocoChorales and URMP. Missing decoder tools now fail before expensive pitch inference. Reports retain full provenance in separate directories. Completed jobs are reused across directories using per-recording input hashes, inference/scoring code, relevant dependencies, and scoring settings; worker counts, method selection, Git HEAD, and unrelated benchmark edits do not invalidate them. Compatible checkpoints from the stopped three-worker run are recognized automatically. Each returned job is atomically checkpointed before the status counter advances. Interruptions preserve completed jobs; with cache-only mode disabled, `force_methods=()` resumes only unfinished or failed jobs. `force=True` explicitly bypasses result reuse. CREPE raw frequency/confidence results are separately cached, so an interrupted note post-processing job can reuse its completed CREPE frontend.


Attune now includes production score alignment, robust timing refitting, and repeat-note recovery on every dataset. Recovered notes are scored against the unchanged references. `refinement_cpu_seconds` records this additional stage and is included in `cpu_seconds`. The optional `attune-audio-only` diagnostic remains score-free and excludes recovery.


## Expanded CocoChorales

This sample may overlap earlier note sweeps; it is a development check. Do not treat the test-split name as proof it was never used for model selection.

In [2]:
expanded_coco = BENCHES["coco"].run_preliminary("coco")
display_metrics(BENCH.summarize(expanded_coco))


Running: preliminary_coco (1040 cached, 520 pending)
[1560/1560] yourmt3: test__woodwind_track229950__4_bassoon      
Reports: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/note/notebook_runs/preliminary_coco_b1b0e714735059fa


,dataset,method,attempted,succeeded,failed,datasets_scored,complete,aggregation,tp,fp,...,precision_offset,recall_offset,Transcription F1,cpu_seconds,wall_seconds,execution_cpu_seconds,frontend_cpu_seconds,segmentation_cpu_seconds,refinement_cpu_seconds,audio_per_cpu
0,coco,Attune,260,260,0,1,True,pooled_note_counts,5889,315,...,0.944230,0.949279,0.946747,2.777809,0.270701,0.239264,2.727964,0.034535,0.01531,7.707719
1,coco,basic-pitch,260,260,0,1,True,pooled_note_counts,5848,3925,...,0.509158,0.806352,0.624185,5.301005,8.008404,7.596056,5.285692,0.015313,NaN,4.038964
2,coco,crepe-notes,260,260,0,1,True,pooled_note_counts,5969,766,...,0.875575,0.955599,0.913839,111.535432,3.735067,3.636211,110.758412,0.777020,NaN,0.191962
3,coco,mt3,260,260,0,1,True,pooled_note_counts,5670,2823,...,0.597904,0.822881,0.692580,14.397003,11.259992,14.397003,NaN,NaN,NaN,1.487155
4,coco,tony,260,260,0,1,True,pooled_note_counts,3673,1962,...,0.611535,0.558418,0.583771,1.394020,1.258300,1.181475,NaN,NaN,NaN,15.358869
5,coco,yourmt3,260,260,0,1,True,pooled_note_counts,5328,556,...,0.796057,0.759034,0.777105,19.561178,20.194717,19.561178,NaN,NaN,NaN,1.094544


## Expanded URMP

Current production pitch settings were selected using the two-per-instrument URMP development sample. These results are not independent held-out evidence. Attune's score-informed row and the audio-only external rows have different input privileges.

In [3]:
expanded_urmp = BENCHES["urmp"].run_preliminary("urmp")
display_metrics(BENCH.summarize(expanded_urmp))


Running: preliminary_urmp (78 cached, 816 pending)
[ 894/894] yourmt3: 44_K515_vn_vn_va_va_vc/AuSep_5_vc_44_K515                  
Reports: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/note/notebook_runs/preliminary_urmp_2670b51fd15f1bb5


,dataset,method,attempted,succeeded,failed,datasets_scored,complete,aggregation,tp,fp,...,precision_offset,recall_offset,Transcription F1,cpu_seconds,wall_seconds,execution_cpu_seconds,frontend_cpu_seconds,segmentation_cpu_seconds,refinement_cpu_seconds,audio_per_cpu
0,urmp,Attune,149,149,0,1,True,pooled_note_counts,26342,3791,...,0.835927,0.854415,0.845070,18.078330,2.627542,2.579869,16.614770,0.123776,1.339784,6.141529
1,urmp,basic-pitch,149,149,0,1,True,pooled_note_counts,25244,9596,...,0.580367,0.685865,0.628722,8.633747,9.641487,9.505769,8.469718,0.164029,NaN,12.859838
2,urmp,crepe-notes,149,149,0,1,True,pooled_note_counts,26078,8421,...,0.678396,0.793867,0.731604,576.479049,19.168360,18.866897,572.594524,3.884525,NaN,0.192598
3,urmp,mt3,149,149,0,1,True,pooled_note_counts,27719,6273,...,0.756502,0.872257,0.810266,40.175639,25.082259,40.175639,NaN,NaN,NaN,2.763580
4,urmp,tony,149,149,0,1,True,pooled_note_counts,13694,2193,...,0.778561,0.419558,0.545274,3.560639,2.955276,2.883481,NaN,NaN,NaN,31.182205
5,urmp,yourmt3,149,149,0,1,True,pooled_note_counts,26932,1908,...,0.864043,0.845256,0.854546,118.919791,121.556337,118.919791,NaN,NaN,NaN,0.933643


## Expanded Bach10 — original recordings

The balanced selection matches `pitch.ipynb`, allowing Attune to reuse compatible production pitch caches. The four instrument parts are evaluated separately; ensemble mixes are excluded. Frame references were generated with YIN and manual corrections, then grouped into notes manually. See the [release description](https://labsites.rochester.edu/air/datasets/Bach10%20Dataset_v1.0.pdf).

This uses the same spawned worker pool, neural worker limit, single updating status line, metrics, and report format as the CocoChorales and URMP cells. The default run has 320 rows (40 recordings × 8 methods), including cached baselines. Results and manifests are saved under a separate `preliminary_bach10-original_*` directory. The sample is a development comparison, not independent held-out evidence.


In [4]:
expanded_bach10 = BENCHES["bach10-original"].run_preliminary("bach10-original")
display_metrics(BENCH.summarize(expanded_bach10))


Running: preliminary_bach10-original (24 cached, 216 pending)
[ 240/240] yourmt3: 10-NunBitten-bassoon                       
Reports: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/note/notebook_runs/preliminary_bach10-original_a4ae8dd11f3819ba


,dataset,method,attempted,succeeded,failed,datasets_scored,complete,aggregation,tp,fp,...,precision_offset,recall_offset,Transcription F1,cpu_seconds,wall_seconds,execution_cpu_seconds,frontend_cpu_seconds,segmentation_cpu_seconds,refinement_cpu_seconds,audio_per_cpu
0,bach10-original,Attune,40,40,0,1,True,pooled_note_counts,1710,228,...,0.828173,0.806938,0.817418,4.918378,0.731060,0.711437,4.507318,0.042258,0.368802,6.799748
1,bach10-original,basic-pitch,40,40,0,1,True,pooled_note_counts,1811,698,...,0.669191,0.844143,0.746554,5.513212,6.848166,6.609696,5.488666,0.024546,NaN,6.066106
2,bach10-original,crepe-notes,40,40,0,1,True,pooled_note_counts,1813,324,...,0.786149,0.844646,0.814348,175.128865,141.716258,140.898819,174.060281,1.068584,NaN,0.190966
3,bach10-original,mt3,40,40,0,1,True,pooled_note_counts,1659,380,...,0.725846,0.744093,0.734856,17.076071,12.398436,17.076071,NaN,NaN,NaN,1.958514
4,bach10-original,tony,40,40,0,1,True,pooled_note_counts,1763,155,...,0.862357,0.831574,0.846685,1.973379,1.516191,1.454217,NaN,NaN,NaN,16.947444
5,bach10-original,yourmt3,40,40,0,1,True,pooled_note_counts,1590,258,...,0.776515,0.721468,0.747980,31.630170,32.110070,31.630170,NaN,NaN,NaN,1.057336


## Paired quality, coverage and instrument breakdown

Differences are pooled-note micro-F1 for Attune minus baseline on successful matching instrument tracks, reported separately per corpus. Exploratory 95% percentile intervals resample whole pieces/tracks (2,000 draws), keeping sibling stems together. They do not establish equivalence: a “comparable” claim needs a prespecified non-inferiority margin and independent evaluation. `complete=False` flags missing pairs. Do not rank methods using unequal coverage without reporting it.

In [5]:
rows = pd.concat(
    [expanded_coco, expanded_urmp, expanded_bach10], ignore_index=True
)
display_metrics(BENCH.summarize(rows))
display_metrics(BENCH.paired_comparisons(rows))
display_metrics(rows.loc[rows.status != "ok", ["dataset", "method", "track_id", "error"]])
good = rows[rows.status == "ok"]
if not good.empty:
    display_metrics(
        pd.concat(
            [
                BENCH.summarize(group).assign(instrument=instrument)
                for (_, instrument), group in good.groupby(["dataset", "instrument"])
            ],
            ignore_index=True,
        )[
            [
                "dataset",
                "instrument",
                "method",
                "succeeded",
                "tp",
                "fp",
                "fn",
                "f1",
                "f1_offset",
            ]
        ]
    )


,dataset,method,attempted,succeeded,failed,datasets_scored,complete,aggregation,tp,fp,...,precision_offset,recall_offset,Transcription F1,cpu_seconds,wall_seconds,execution_cpu_seconds,frontend_cpu_seconds,segmentation_cpu_seconds,refinement_cpu_seconds,audio_per_cpu
0,bach10-original,Attune,40,40,0,1,True,pooled_note_counts,1710,228,...,0.828173,0.806938,0.817418,4.918378,0.731060,0.711437,4.507318,0.042258,0.368802,6.799748
1,bach10-original,basic-pitch,40,40,0,1,True,pooled_note_counts,1811,698,...,0.669191,0.844143,0.746554,5.513212,6.848166,6.609696,5.488666,0.024546,NaN,6.066106
2,bach10-original,crepe-notes,40,40,0,1,True,pooled_note_counts,1813,324,...,0.786149,0.844646,0.814348,175.128865,141.716258,140.898819,174.060281,1.068584,NaN,0.190966
3,bach10-original,mt3,40,40,0,1,True,pooled_note_counts,1659,380,...,0.725846,0.744093,0.734856,17.076071,12.398436,17.076071,NaN,NaN,NaN,1.958514
4,bach10-original,tony,40,40,0,1,True,pooled_note_counts,1763,155,...,0.862357,0.831574,0.846685,1.973379,1.516191,1.454217,NaN,NaN,NaN,16.947444
5,bach10-original,yourmt3,40,40,0,1,True,pooled_note_counts,1590,258,...,0.776515,0.721468,0.747980,31.630170,32.110070,31.630170,NaN,NaN,NaN,1.057336
6,coco,Attune,260,260,0,1,True,pooled_note_counts,5889,315,...,0.944230,0.949279,0.946747,2.777809,0.270701,0.239264,2.727964,0.034535,0.015310,7.707719
7,coco,basic-pitch,260,260,0,1,True,pooled_note_counts,5848,3925,...,0.509158,0.806352,0.624185,5.301005,8.008404,7.596056,5.285692,0.015313,NaN,4.038964
8,coco,crepe-notes,260,260,0,1,True,pooled_note_counts,5969,766,...,0.875575,0.955599,0.913839,111.535432,3.735067,3.636211,110.758412,0.777020,NaN,0.191962
9,coco,mt3,260,260,0,1,True,pooled_note_counts,5670,2823,...,0.597904,0.822881,0.692580,14.397003,11.259992,14.397003,NaN,NaN,NaN,1.487155


,dataset,baseline,metric,aggregation,paired_tracks,groups,delta,ci_low,ci_high,complete
0,bach10-original,basic-pitch,Onset F1,pooled_note_counts,40,10,0.065647,0.045345,0.089328,True
1,bach10-original,crepe-notes,Onset F1,pooled_note_counts,40,10,-0.007923,-0.032516,0.017369,True
2,bach10-original,tony,Onset F1,pooled_note_counts,40,10,-0.031589,-0.049637,-0.013338,True
3,bach10-original,mt3,Onset F1,pooled_note_counts,40,10,0.047160,0.029412,0.062000,True
4,bach10-original,yourmt3,Onset F1,pooled_note_counts,40,10,0.042121,0.025389,0.060860,True
5,bach10-original,basic-pitch,Transcription F1,pooled_note_counts,40,10,0.070864,0.035785,0.105434,True
6,bach10-original,crepe-notes,Transcription F1,pooled_note_counts,40,10,0.003070,-0.043711,0.042597,True
7,bach10-original,tony,Transcription F1,pooled_note_counts,40,10,-0.029268,-0.060588,0.001774,True
8,bach10-original,mt3,Transcription F1,pooled_note_counts,40,10,0.082562,0.043327,0.114878,True
9,bach10-original,yourmt3,Transcription F1,pooled_note_counts,40,10,0.069438,0.037167,0.099043,True


,dataset,method,track_id,error


,dataset,instrument,method,succeeded,tp,fp,fn,Onset F1,Transcription F1
0,bach10-original,bassoon,Attune,10,524,33,35,0.939068,0.917563
1,bach10-original,bassoon,basic-pitch,10,513,356,46,0.718487,0.693277
2,bach10-original,bassoon,crepe-notes,10,529,84,30,0.902730,0.854949
3,bach10-original,bassoon,mt3,10,512,77,47,0.891986,0.766551
4,bach10-original,bassoon,tony,10,530,28,29,0.948970,0.927484
...,...,...,...,...,...,...,...,...,...
175,urmp,violin,basic-pitch,34,6749,1803,857,0.835376,0.622602
176,urmp,violin,crepe-notes,34,6968,1563,638,0.863605,0.773254
177,urmp,violin,mt3,34,7193,1354,413,0.890609,0.827215
178,urmp,violin,tony,34,2516,502,5090,0.473645,0.407380


## Competitor follow-up and claim boundary

The historical results include Basic Pitch, CREPE Notes and Tony plus PELT (L1/L2/RBF), KernelCPD (linear/Gaussian/cosine), bottom-up and window segmentation; a one-track neural-boundary experiment is not a meaningful competitor benchmark. Current configuration also lists oracle-count Dynp and slope-aware windows. Official JAX MT3 (`mt3`) is included by default; `mt3-port.ipynb` provides setup and bounded validation.

Further candidates: [YourMT3+](https://arxiv.org/abs/2407.04822) for stronger multi-instrument transcription, plus the newly released [MuScriptor (2026)](https://arxiv.org/abs/2607.08168); audit checkpoint training overlap and use appropriate URMP held-out splits before evaluating them. [Melodia → notes](https://github.com/justinsalamon/audio_to_midi_melodia) is a useful classical sanity baseline, but its authors describe the segmenter as deliberately simple. Tony is already represented and should remain in the default experiment. Piano-only transcription systems are not direct replacements for this varied-instrument suite.

The historical descriptive result is: “In the saved CocoChorales benchmark, the linear KernelCPD configuration achieved higher mean onset-and-pitch F1 than Basic Pitch and CREPE Notes on the same 2,486 stems.” A present-tense claim about current Attune requires the fresh results, explicit input/metric conditions, and preferably an untouched evaluation set. Neither notebook establishes performance on mixtures, live microphone recordings, or unseen instruments.

## Overall metrics across datasets

Pool true positives, false positives, and false negatives across all successful instrument tracks in all three datasets, then compute micro precision, recall, and F1. Matching is performed separately within each recording. **Onset F1** requires matching pitch and onset (50 cents / 50 ms), ignoring offsets. **Transcription F1** additionally requires the offset within max(50 ms, 20% of reference duration). Both are pooled micro-F1; the overall table ranks by Transcription F1. Failed recordings remain reported as failures, not zero scores; `complete` flags incomplete coverage. CPU time remains an average per successful recording, and `audio_per_cpu` divides total audio duration by total algorithm CPU time.


In [6]:
overall_rows = pd.concat(
    [expanded_coco, expanded_urmp, expanded_bach10], ignore_index=True
)
# Never publish an incomplete 449-track table as the expanded evaluation.
expected_ids = {(dataset, track["track_id"]) for dataset, tracks in selected_tracks.items() for track in tracks}
for method in CONFIG.methods:
    method_rows = overall_rows.loc[overall_rows.method == method]
    actual_ids = set(zip(method_rows.dataset, method_rows.track_id))
    if len(method_rows) != 449 or actual_ids != expected_ids or not method_rows.status.eq("ok").all():
        raise RuntimeError(f"{method}: incomplete expanded evaluation; inspect the failure report and rerun to resume.")
overall_summary = BENCH.summarize(overall_rows, by_dataset=False).set_index("method")
quality_metrics = [
    "precision",
    "recall",
    "f1",
    "precision_offset",
    "recall_offset",
    "f1_offset",
]
overall_summary[quality_metrics] *= 100
overall_summary = (
    overall_summary[
        [
            "attempted",
            "succeeded",
            "failed",
            "datasets_scored",
            "complete",
            "tp",
            "fp",
            "fn",
            *quality_metrics[:3],
            "tp_offset",
            "fp_offset",
            "fn_offset",
            *quality_metrics[3:],
            "cpu_seconds",
            "audio_per_cpu",
        ]
    ]
    .rename(
        columns={
            "precision": "Onset precision (%)",
            "recall": "Onset recall (%)",
            "f1": "Onset F1 (%)",
            "precision_offset": "Transcription precision (%)",
            "recall_offset": "Transcription recall (%)",
            "f1_offset": "Transcription F1 (%)",
            "cpu_seconds": "CPU seconds / recording",
        }
    )
    .sort_values("Transcription F1 (%)", ascending=False)
)
display(
    overall_summary.rename(index={"attune": "Attune"}).style.format(
        {
            column: "{:.2f}"
            for column in overall_summary.columns
            if "(%)" in column or column in ("CPU seconds / recording", "audio_per_cpu")
        },
        na_rep="—",
    )
)


,attempted,succeeded,failed,datasets_scored,complete,tp,fp,fn,Onset precision (%),Onset recall (%),Onset F1 (%),tp_offset,fp_offset,fn_offset,Transcription precision (%),Transcription recall (%),Transcription F1 (%),CPU seconds / recording,audio_per_cpu
method,,,,,,,,,,,,,,,,,,,
Attune,449,449,0,3,True,33941,4334,3700,88.68,90.17,89.42,32652,5623,4989,85.31,86.75,86.02,8.05,6.49
yourmt3,449,449,0,3,True,33850,2722,3791,92.56,89.93,91.22,31038,5534,6603,84.87,82.46,83.65,53.61,0.97
mt3,449,449,0,3,True,35048,9476,2593,78.72,93.11,85.31,32273,12251,5368,72.48,85.74,78.56,23.19,2.25
crepe-notes,449,449,0,3,True,33860,9511,3781,78.07,89.96,83.59,30981,12390,6660,71.43,82.31,76.48,271.49,0.19
basic-pitch,449,449,0,3,True,32903,14219,4738,69.83,87.41,77.64,26875,20247,10766,57.03,71.40,63.41,6.43,8.13
tony,449,449,0,3,True,19130,4310,18511,81.61,50.82,62.64,17469,5971,20172,74.53,46.41,57.20,2.16,24.13


## Paired significance of overall Transcription F1

Recompute two-sided paired permutation tests from `overall_rows` above, without rerunning inference. The default family is Attune versus Basic Pitch, CREPE Notes, Tony, MT3, and YourMT3+ on **Transcription F1 (pitch + onset + offset micro-F1)**, with Holm correction across the five comparisons at 0.05. Positive differences favor Attune. Adding `"f1"` to `SIGNIFICANCE_METRICS` includes those comparisons in the same Holm family.

All comparisons use the common successful track intersection. Swaps keep whole source groups together, conservatively merging repeated URMP titles, and recompute pooled F1 from swapped TP/FP/FN counts. Small designs enumerate every swap; larger designs use 9,999 random swaps with a plus-one correction. Coverage reports excluded tracks; failed runs are never assigned zero scores. `SOURCE_GROUPS` can link known shared compositions across datasets. The same cell supports larger runs by consuming their rows from the preceding overall-results cell; significance does not establish held-out generalization or superiority for every instrument.


In [7]:
import benchmarks.modules.note.NoteBenchmarker as note_significance_module
importlib.reload(note_significance_module)
SIGNIFICANCE_METHODS = CONFIG.methods
SIGNIFICANCE_METRICS = ('f1_offset',)
SIGNIFICANCE_RESAMPLES = 9999
SIGNIFICANCE_ALPHA = 0.05
SOURCE_GROUPS = {}
note_significance, significance_coverage = note_significance_module.NoteEvaluation.paired_significance(overall_rows, methods=SIGNIFICANCE_METHODS, metrics=SIGNIFICANCE_METRICS, n_resamples=SIGNIFICANCE_RESAMPLES, seed=CONFIG.seed, alpha=SIGNIFICANCE_ALPHA, source_groups=SOURCE_GROUPS)
display(significance_coverage)
significance_display = note_significance.rename(columns={
    'attune_f1': 'Attune Transcription F1 (%)',
    'competitor_f1': 'Competitor Transcription F1 (%)',
}).replace({'metric': METRIC_LABELS})
display(significance_display.style.format({'Attune Transcription F1 (%)': '{:.2f}', 'Competitor Transcription F1 (%)': '{:.2f}', 'difference_pp': '{:+.2f}', 'p_value': '{:.4g}', 'p_adj': '{:.4g}', 'alpha': '{:.2f}'}))
print('Transcription F1 columns are percentages; difference_pp is Attune minus competitor.')
print('significant tests p_adj <= alpha (two-sided); inspect difference_pp for direction.')


,dataset,method,selected_tracks,attempted,succeeded,paired_tracks,excluded_tracks
0,bach10-original,attune,40,40,40,40,0
1,bach10-original,basic-pitch,40,40,40,40,0
2,bach10-original,crepe-notes,40,40,40,40,0
3,bach10-original,tony,40,40,40,40,0
4,bach10-original,mt3,40,40,40,40,0
5,bach10-original,yourmt3,40,40,40,40,0
6,coco,attune,260,260,260,260,0
7,coco,basic-pitch,260,260,260,260,0
8,coco,crepe-notes,260,260,260,260,0
9,coco,tony,260,260,260,260,0


,method,metric,tracks,source_groups,excluded_tracks,Attune Transcription F1 (%),Competitor Transcription F1 (%),difference_pp,p_value,exact,permutations,seed,p_adj,significant,alpha
0,basic-pitch,Transcription F1,449,293,0,86.02,63.41,+22.61,0.0001,False,9999,0,0.0005,True,0.05
1,crepe-notes,Transcription F1,449,293,0,86.02,76.48,+9.54,0.0001,False,9999,0,0.0005,True,0.05
2,tony,Transcription F1,449,293,0,86.02,57.20,+28.82,0.0001,False,9999,0,0.0005,True,0.05
3,mt3,Transcription F1,449,293,0,86.02,78.56,+7.46,0.0001,False,9999,0,0.0005,True,0.05
4,yourmt3,Transcription F1,449,293,0,86.02,83.65,+2.38,0.009,False,9999,0,0.009,True,0.05


Transcription F1 columns are percentages; difference_pp is Attune minus competitor.
significant tests p_adj <= alpha (two-sided); inspect difference_pp for direction.


## Paired significance of overall Onset F1

Run after the Transcription F1 significance cell above. Compare Attune with each competitor on **Onset F1 (pitch + onset micro-F1, ignoring offsets)** using the same overall rows, source groups, seed, and permutation settings, without rerunning inference. Holm correction is applied separately across the five onset comparisons at 0.05; it does not jointly correct onset and transcription comparisons. To test both metrics as one family, use `SIGNIFICANCE_METRICS = ('f1_offset', 'f1')` in the preceding cell.

Positive `difference_pp` favors Attune; negative values favor the competitor. The `significant` column indicates whether the two-sided Holm-adjusted p-value is at or below alpha. Coverage reports the common successful track intersection and excluded tracks.


In [8]:
import benchmarks.modules.note.NoteBenchmarker as note_significance_module
importlib.reload(note_significance_module)
ONSET_SIGNIFICANCE_METRICS = ('f1',)
onset_significance, onset_significance_coverage = note_significance_module.NoteEvaluation.paired_significance(overall_rows, methods=SIGNIFICANCE_METHODS, metrics=ONSET_SIGNIFICANCE_METRICS, n_resamples=SIGNIFICANCE_RESAMPLES, seed=CONFIG.seed, alpha=SIGNIFICANCE_ALPHA, source_groups=SOURCE_GROUPS)
display(onset_significance_coverage)
onset_significance_display = onset_significance.rename(columns={
    'attune_f1': 'Attune Onset F1 (%)',
    'competitor_f1': 'Competitor Onset F1 (%)',
}).replace({'metric': METRIC_LABELS})
display(onset_significance_display.style.format({'Attune Onset F1 (%)': '{:.2f}', 'Competitor Onset F1 (%)': '{:.2f}', 'difference_pp': '{:+.2f}', 'p_value': '{:.4g}', 'p_adj': '{:.4g}', 'alpha': '{:.2f}'}))
print('Onset F1 columns are percentages; difference_pp is Attune minus competitor.')
print('significant tests p_adj <= alpha (two-sided); inspect difference_pp for direction.')


,dataset,method,selected_tracks,attempted,succeeded,paired_tracks,excluded_tracks
0,bach10-original,attune,40,40,40,40,0
1,bach10-original,basic-pitch,40,40,40,40,0
2,bach10-original,crepe-notes,40,40,40,40,0
3,bach10-original,tony,40,40,40,40,0
4,bach10-original,mt3,40,40,40,40,0
5,bach10-original,yourmt3,40,40,40,40,0
6,coco,attune,260,260,260,260,0
7,coco,basic-pitch,260,260,260,260,0
8,coco,crepe-notes,260,260,260,260,0
9,coco,tony,260,260,260,260,0


,method,metric,tracks,source_groups,excluded_tracks,Attune Onset F1 (%),Competitor Onset F1 (%),difference_pp,p_value,exact,permutations,seed,p_adj,significant,alpha
0,basic-pitch,Onset F1,449,293,0,89.42,77.64,+11.78,0.0001,False,9999,0,0.0005,True,0.05
1,crepe-notes,Onset F1,449,293,0,89.42,83.59,+5.82,0.0001,False,9999,0,0.0005,True,0.05
2,tony,Onset F1,449,293,0,89.42,62.64,+26.78,0.0001,False,9999,0,0.0005,True,0.05
3,mt3,Onset F1,449,293,0,89.42,85.31,+4.11,0.0025,False,9999,0,0.005,True,0.05
4,yourmt3,Onset F1,449,293,0,89.42,91.22,-1.81,0.2038,False,9999,0,0.2038,False,0.05


Onset F1 columns are percentages; difference_pp is Attune minus competitor.
significant tests p_adj <= alpha (two-sided); inspect difference_pp for direction.
